# TCC — continuar a base original até a época 300

Anexe TCC RGI Guaxupé v1 e o backup completo original das 100 épocas como entradas privadas. Ative Internet e GPU T4. A U-Net roda primeiro, seguida da U-Mamba. Cada modelo recebe 200 épocas adicionais. O teste antigo fica reservado.

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone

CODE_REF = "19d42d664ddf95114cff1115b8cf2ed71d4e4274"
WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = WORK / "tcc_continuacao_300"
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/oguel/tcc-umamba.git", str(REPO)], check=True)
if subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip():
    raise RuntimeError("Preserve as alterações no checkout antes de continuar.")
subprocess.run(["git", "fetch", "origin", CODE_REF], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", CODE_REF], cwd=REPO, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == CODE_REF
print("Código fixado:", CODE_REF)


In [ ]:
EXPECTED = {
    "unet": "addd3b5f680c1495dfa73b7eae6332fe085015c8d3e7035d825dfab4adc1d2cf",
    "umamba": "038a100adc397ae7482bf12550f958c9046829d5ff3d4f16d56275f69a91f9bf",
}
def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

roots = [WORK / "tcc_runs", INPUT, WORK / "backup_final_extraido"]
def locate():
    runs = {}
    for kind, expected in EXPECTED.items():
        candidates = [p for root in roots if root.exists() for p in root.rglob(f"best_{kind}_rgi.pt")]
        for path in sorted(candidates):
            if sha256(path) == expected:
                runs[kind] = path.parent
                break
    return runs

RUNS = locate()
if len(RUNS) != 2:
    candidates = [p for p in INPUT.rglob("*.zip") if "backup" in p.name.lower() and p.stat().st_size == 53710616]
    complete = [p for p in candidates if sha256(p) == "234cb97b21931b4c3b9ec8140003b17ca5319a03cfa899b297140160a6b61a1a"]
    if complete:
        destination = WORK / "backup_final_extraido"
        if destination.exists():
            raise RuntimeError("Extração anterior incompleta: preserve a pasta e revise o problema.")
        with zipfile.ZipFile(complete[0]) as archive:
            entries = archive.infolist()
            if len(entries) > 500 or sum(e.file_size for e in entries) > 512 * 1024**2:
                raise RuntimeError("Dimensão do arquivo inesperada.")
            for entry in entries:
                resolved = (destination / entry.filename).resolve()
                if not resolved.is_relative_to(destination.resolve()):
                    raise RuntimeError("Caminho inválido no ZIP.")
            if archive.testzip() is not None:
                raise RuntimeError("ZIP corrompido.")
            destination.mkdir()
            archive.extractall(destination)
        RUNS = locate()
if len(RUNS) != 2:
    raise RuntimeError("Anexe como entrada privada o backup_COMPLETO_UNET_UMAMBA.zip ou as duas pastas de execução completas.")
print({kind: str(path) for kind, path in RUNS.items()})


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements-runtime.txt"), "geopandas"], check=True)
subprocess.run([sys.executable, "-c", "from src.models.umamba_runtime import install_prebuilt_colab_stack; print('PyTorch alterado:', install_prebuilt_colab_stack())"], cwd=REPO, check=True)


In [ ]:
DATA_ROOT = INPUT
image_folders = list(DATA_ROOT.rglob("Imagens_Individuais"))
if not image_folders:
    import hashlib
    import zipfile
    archives = list(DATA_ROOT.rglob("dataset_rgi_v1_privado.zip"))
    if len(archives) != 1:
        raise RuntimeError("Anexe o dataset privado com os GeoTIFFs ou o ZIP preparado.")
    digest = hashlib.sha256()
    with archives[0].open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    assert digest.hexdigest() == "9cae74a47fcc68885f2703a5bf3cd5d9230eadc400a9c5a57c3ed751376d60b4", "ZIP diferente"
    destination = WORK / "dataset_rgi_v1"
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for name in archive.namelist():
            if not (destination / name).resolve().is_relative_to(destination.resolve()):
                raise ValueError("Caminho inválido no ZIP")
        archive.extractall(destination)
    DATA_ROOT = destination
print("Raiz de dados:", DATA_ROOT)


In [ ]:
import os

EPOCHS = 300
EXPECTED_LAST = {
    "unet": "65a9617ab8e600daf3e7ec01ed236383f4af6f92a6546d3ec574eca388695ad4",
    "umamba": "a7fd5321815cc3414ff32619363e7daA03db8389d9ef9d7dfaca38882493d31f".lower(),
}
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output = WORK / f"tcc_base_original_300_{stamp}"
for model, source in RUNS.items():
    if sha256(source / "last.pt") != EXPECTED_LAST[model]:
        raise RuntimeError(f"Checkpoint da epoca 100 divergente para {model}.")
    shutil.copytree(source, output / model)
wrapper = output / "continuar_treino.py"
wrapper.write_text('from pathlib import Path\n\noriginal = Path("scripts/train_emater_rgi.py").resolve()\nsource = original.read_text(encoding="utf-8")\nline = \'"epoch_budget": epochs,\'\nif source.count(line) != 1:\n    raise RuntimeError("Treinador original divergiu do esperado.")\n# O contrato dos checkpoints continua fixado em 100.\n# O argumento --epochs controla o novo limite real do loop.\nsource = source.replace(line, \'"epoch_budget": 100,\', 1)\nscope = {"__name__": "__main__", "__file__": str(original), "__package__": "scripts"}\nexec(compile(source, str(original), "exec"), scope)\n', encoding="utf-8")
(output / "extensao.json").write_text(json.dumps({
    "source_revision": CODE_REF, "initial_epoch": 100, "target_epoch": EPOCHS,
    "source_last_sha256": EXPECTED_LAST, "wrapper_sha256": sha256(wrapper),
    "test_evaluated": False,
}, indent=2), encoding="utf-8")
for model in ("unet", "umamba"):
    print(f"Retomando {model} a partir da epoca 101", flush=True)
    subprocess.run([sys.executable, "-u", str(wrapper),
        "--root", str(DATA_ROOT), "--model", model, "--device", "cuda",
        "--run-mode", "training", "--epochs", str(EPOCHS), "--batch-size", "1",
        "--patch-size", "256", "--augment", "--seed", "42",
        "--train-views-per-image", "4", "--num-workers", "0", "--output", str(output),
        "--resume-run", str(output / model)],
        cwd=REPO, env={**os.environ, "PYTHONPATH": str(REPO)}, check=True)
    status = json.loads((output / model / "status.json").read_text())
    assert status["epochs_completed"] == EPOCHS and not status["blind_test_evaluated"]
    print("BACKUP:", shutil.make_archive(str(output) + "_" + model, "zip", root_dir=output))
